
# 19 — Profile Similarity Distance  
## Pairwise Distance Across Final Five-Variable Country Profiles

This notebook adds the similarity-distance component Francesco asked about.

It should be run **after Notebook 08**:

```text
08_Profile_POSet_Main_2002_5Var.ipynb
```

## What this computes

The POSet tells us whether one country/profile dominates another.  
The similarity-distance analysis tells us how close two country profiles are, including cases that are incomparable under Pareto dominance.

The main distance is computed from the final five ordinal profile variables:

```text
debt capacity
employment strength
R&D intensity
tertiary human capital
energy security
```

For two countries \(i\) and \(j\):

\[
d(i,j)=\frac{\sum_{k=1}^{5}|x_{ik}-x_{jk}|}{5(5-1)}
\]

Similarity is:

\[
s(i,j)=1-d(i,j)
\]

This complements the POSet. It does **not** create a scalar Economic Resilience Index.


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_Similarity_Distance"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "19_Profile_Similarity_Distance"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Profile_Similarity_Distance"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Profile_Similarity_Distance"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, FIGURES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())


Run ID: 20260625_094100
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

MAIN_VARIABLE_SET_NAME = "baseline_5_variables"
MAIN_LEVELS = 5

LEVEL_COLUMNS = [
    "debt_capacity_level_5",
    "employment_strength_level_5",
    "rd_intensity_level_5",
    "human_capital_tertiary_level_5",
    "energy_security_level_5",
]

ALT_LEVEL_COLUMNS = [
    "debt_capacity_score_0_100__L5",
    "employment_strength_score_0_100__L5",
    "rd_intensity_score_0_100__L5",
    "human_capital_tertiary_score_0_100__L5",
    "energy_security_score_0_100__L5",
]

SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

VARIABLE_LABELS = {
    "debt_capacity_level_5": "Debt capacity",
    "employment_strength_level_5": "Employment strength",
    "rd_intensity_level_5": "R&D intensity",
    "human_capital_tertiary_level_5": "Tertiary human capital",
    "energy_security_level_5": "Energy security",
    "debt_capacity_score_0_100__L5": "Debt capacity",
    "employment_strength_score_0_100__L5": "Employment strength",
    "rd_intensity_score_0_100__L5": "R&D intensity",
    "human_capital_tertiary_score_0_100__L5": "Tertiary human capital",
    "energy_security_score_0_100__L5": "Energy security",
}

MAX_ORDINAL_DISTANCE = len(LEVEL_COLUMNS) * (MAIN_LEVELS - 1)

print("Main variable set:", MAIN_VARIABLE_SET_NAME)
print("Main levels:", MAIN_LEVELS)
print("Max raw ordinal distance:", MAX_ORDINAL_DISTANCE)


Main variable set: baseline_5_variables
Main levels: 5
Max raw ordinal distance: 20


In [3]:

# ------------------------------------------------------
# HELPERS
# ------------------------------------------------------

table_inventory_rows = []
figure_inventory_rows = []

def find_first_existing(candidates, label):
    for p in candidates:
        if p.exists():
            print(f"Using {label}: {p}")
            return p
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )

def clean_keys(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    if "baseline_year" in out.columns:
        out["baseline_year"] = pd.to_numeric(out["baseline_year"], errors="coerce").astype("Int64")

    return out

def save_table(df, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    df.to_csv(processed_path, index=False)
    df.to_csv(diagnostics_path, index=False)
    df.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved table:", file_name)

def save_matrix(df, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    df.to_csv(processed_path)
    df.to_csv(diagnostics_path)
    df.to_csv(table_path)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved matrix:", file_name)

def save_figure(fig, file_name, title, description):
    path = FIGURES_DIR / file_name
    fig.savefig(path, dpi=260, bbox_inches="tight")
    plt.close(fig)

    figure_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "path": str(path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved figure:", file_name)

def pick_level_columns(df):
    if all(c in df.columns for c in LEVEL_COLUMNS):
        return LEVEL_COLUMNS

    if all(c in df.columns for c in ALT_LEVEL_COLUMNS):
        return ALT_LEVEL_COLUMNS

    candidates = [c for c in df.columns if c.endswith("__L5") or c.endswith("_level_5")]
    candidates = [
        c for c in candidates
        if "governance" not in c.lower()
        and "volatility" not in c.lower()
        and "productivity" not in c.lower()
        and any(k in c.lower() for k in ["debt", "employment", "rd", "tertiary", "energy", "human_capital"])
    ]

    ordered = []
    for key in ["debt", "employment", "rd_", "human_capital", "energy"]:
        matches = [c for c in candidates if key in c.lower()]
        if matches:
            ordered.append(matches[0])

    if len(ordered) == 5:
        return ordered

    raise KeyError(
        "Could not identify the five final ordinal level columns. "
        "Expected either LEVEL_COLUMNS or ALT_LEVEL_COLUMNS."
    )


In [4]:

# ------------------------------------------------------
# LOAD COUNTRY-PROFILE INPUT DATA
# ------------------------------------------------------

PROFILE_COUNTRY_MAP_FILE = find_first_existing(
    [
        PROFILE_DIR / "profile_country_map_with_layers.csv",
        PROFILE_DIR / "profile_country_map.csv",
        PRE_POSET_DIR / "main_profile_discretization_5levels.csv",
        MASTER_DIR / "structural_snapshot_final5_complete_cases.csv",
    ],
    "country profile data",
)

country_profiles_raw = clean_keys(pd.read_csv(PROFILE_COUNTRY_MAP_FILE))

print("Input shape:", country_profiles_raw.shape)
print("Input columns:")
print(country_profiles_raw.columns.tolist())

level_columns = pick_level_columns(country_profiles_raw)
print("\nUsing level columns:")
for c in level_columns:
    print("-", c)

required_id_cols = ["Country", "shock_id"]
missing_required = [c for c in required_id_cols if c not in country_profiles_raw.columns]
if missing_required:
    raise KeyError(f"Missing required ID columns: {missing_required}")

country_profiles = country_profiles_raw.copy()

# Filter final profile configuration if these columns exist.
if "main_variable_set" in country_profiles.columns:
    country_profiles = country_profiles[country_profiles["main_variable_set"].astype(str) == MAIN_VARIABLE_SET_NAME].copy()

if "levels" in country_profiles.columns:
    country_profiles = country_profiles[pd.to_numeric(country_profiles["levels"], errors="coerce") == MAIN_LEVELS].copy()

if "profile_levels" in country_profiles.columns:
    country_profiles = country_profiles[pd.to_numeric(country_profiles["profile_levels"], errors="coerce") == MAIN_LEVELS].copy()

if "set_name" in country_profiles.columns:
    if (country_profiles["set_name"].astype(str) == MAIN_VARIABLE_SET_NAME).any():
        country_profiles = country_profiles[country_profiles["set_name"].astype(str) == MAIN_VARIABLE_SET_NAME].copy()

if "country_name" not in country_profiles.columns:
    country_profiles["country_name"] = country_profiles["Country"]

for c in level_columns:
    country_profiles[c] = pd.to_numeric(country_profiles[c], errors="coerce")

country_profiles = country_profiles.dropna(subset=level_columns).copy()
country_profiles = country_profiles.sort_values(["shock_id", "Country"]).drop_duplicates(["shock_id", "Country"], keep="first").copy()

print("\nFiltered country-profile shape:", country_profiles.shape)
print(country_profiles.groupby("shock_id")["Country"].nunique())

display(country_profiles[["shock_id", "Country", "country_name"] + level_columns].head())


Using country profile data: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Profile_POSet_Main\profile_country_map_with_layers.csv
Input shape: (60, 19)
Input columns:
['Country', 'shock_id', 'baseline_year', 'profile_code', 'profile_digit_sum', 'debt_capacity_level_5', 'employment_strength_level_5', 'rd_intensity_level_5', 'human_capital_tertiary_level_5', 'energy_security_level_5', 'debt_capacity_score_0_100', 'employment_strength_score_0_100', 'rd_intensity_score_0_100', 'human_capital_tertiary_score_0_100', 'energy_security_score_0_100', 'main_variable_set', 'levels', 'layer', 'is_pareto_frontier']

Using level columns:
- debt_capacity_level_5
- employment_strength_level_5
- rd_intensity_level_5
- human_capital_tertiary_level_5
- energy_security_level_5

Filtered country-profile shape: (60, 20)
shock_id
2007    25
2019    35
Name: Country, dtype: int64


,shock_id,Country,country_name,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5
0,2007,AUT,AUT,2,4,5,3,2
1,2007,BEL,BEL,1,2,4,4,1
2,2007,CAN,CAN,3,3,4,5,5
3,2007,CZE,CZE,4,3,3,1,5
4,2007,DEU,DEU,2,1,4,2,3


In [5]:

# ------------------------------------------------------
# COMPUTE PAIRWISE ORDINAL AND CONTINUOUS DISTANCES
# ------------------------------------------------------

pairwise_rows = []

for shock_id, df_shock in country_profiles.groupby("shock_id"):
    df_shock = df_shock.sort_values("Country").reset_index(drop=True)
    countries = df_shock["Country"].tolist()

    X_ord = df_shock[level_columns].to_numpy(dtype=float)

    score_cols_available = [c for c in SCORE_COLUMNS if c in df_shock.columns]
    X_score = None
    if len(score_cols_available) == len(SCORE_COLUMNS):
        X_score = df_shock[score_cols_available].to_numpy(dtype=float)

    for i, j in itertools.combinations(range(len(df_shock)), 2):
        c1 = countries[i]
        c2 = countries[j]

        diff_ord = np.abs(X_ord[i] - X_ord[j])
        manhattan_raw = float(diff_ord.sum())
        manhattan_norm = manhattan_raw / MAX_ORDINAL_DISTANCE
        similarity = 1 - manhattan_norm
        euclidean_ord = float(np.sqrt((diff_ord ** 2).sum()))
        hamming_count = int((diff_ord > 0).sum())

        row = {
            "shock_id": shock_id,
            "country_a": c1,
            "country_b": c2,
            "country_pair": f"{c1}-{c2}",
            "ordinal_manhattan_distance_raw": manhattan_raw,
            "ordinal_manhattan_distance_0_1": manhattan_norm,
            "ordinal_similarity_0_1": similarity,
            "ordinal_euclidean_distance": euclidean_ord,
            "dimensions_different_count": hamming_count,
            "dimensions_same_count": len(level_columns) - hamming_count,
        }

        for idx, col in enumerate(level_columns):
            label = VARIABLE_LABELS.get(col, col)
            row[f"abs_diff_{label}"] = diff_ord[idx]

        if X_score is not None:
            diff_score = np.abs(X_score[i] - X_score[j])
            cont_euclidean = float(np.sqrt((diff_score ** 2).sum()))
            cont_euclidean_norm = cont_euclidean / (100 * np.sqrt(len(SCORE_COLUMNS)))
            row["continuous_score_euclidean_distance_0_1"] = cont_euclidean_norm
            row["continuous_score_similarity_0_1"] = 1 - cont_euclidean_norm

        pairwise_rows.append(row)

country_profile_distance_long = pd.DataFrame(pairwise_rows)

save_table(
    country_profile_distance_long,
    "country_profile_distance_long.csv",
    "Country profile pairwise distance long table",
    "Pairwise ordinal profile distances and similarities between countries by shock.",
)

display(country_profile_distance_long.head())


Saved table: country_profile_distance_long.csv


,shock_id,country_a,country_b,country_pair,ordinal_manhattan_distance_raw,ordinal_manhattan_distance_0_1,ordinal_similarity_0_1,ordinal_euclidean_distance,dimensions_different_count,dimensions_same_count,abs_diff_Debt capacity,abs_diff_Employment strength,abs_diff_R&D intensity,abs_diff_Tertiary human capital,abs_diff_Energy security,continuous_score_euclidean_distance_0_1,continuous_score_similarity_0_1
0,2007,AUT,BEL,AUT-BEL,6.0000,0.3000,0.7000,2.8284,5,0,1.0000,2.0000,1.0000,1.0000,1.0000,0.2284,0.7716
1,2007,AUT,CAN,AUT-CAN,8.0000,0.4000,0.6000,4.0000,5,0,1.0000,1.0000,1.0000,2.0000,3.0000,0.4850,0.5150
2,2007,AUT,CZE,AUT-CZE,10.0000,0.5000,0.5000,4.6904,5,0,2.0000,1.0000,2.0000,2.0000,3.0000,0.3132,0.6868
3,2007,AUT,DEU,AUT-DEU,6.0000,0.3000,0.7000,3.4641,4,1,0.0000,3.0000,1.0000,1.0000,1.0000,0.2232,0.7768
4,2007,AUT,DNK,AUT-DNK,7.0000,0.3500,0.6500,3.8730,4,1,2.0000,1.0000,0.0000,1.0000,3.0000,0.3340,0.6660


In [6]:

# ------------------------------------------------------
# CREATE DISTANCE AND SIMILARITY MATRICES BY SHOCK
# ------------------------------------------------------

distance_matrices = {}
similarity_matrices = {}

for shock_id, df_shock in country_profiles.groupby("shock_id"):
    countries = sorted(df_shock["Country"].unique())

    D = pd.DataFrame(0.0, index=countries, columns=countries)
    S = pd.DataFrame(1.0, index=countries, columns=countries)

    pairs = country_profile_distance_long[country_profile_distance_long["shock_id"].astype(str) == str(shock_id)]

    for _, row in pairs.iterrows():
        a = row["country_a"]
        b = row["country_b"]
        d = row["ordinal_manhattan_distance_0_1"]
        s = row["ordinal_similarity_0_1"]

        D.loc[a, b] = d
        D.loc[b, a] = d
        S.loc[a, b] = s
        S.loc[b, a] = s

    distance_matrices[str(shock_id)] = D
    similarity_matrices[str(shock_id)] = S

    save_matrix(
        D,
        f"country_profile_distance_matrix_shock_{shock_id}.csv",
        f"Country profile distance matrix shock {shock_id}",
        "Normalized ordinal Manhattan distance matrix.",
    )

    save_matrix(
        S,
        f"country_profile_similarity_matrix_shock_{shock_id}.csv",
        f"Country profile similarity matrix shock {shock_id}",
        "Ordinal profile similarity matrix where similarity = 1 - distance.",
    )

print("Matrices created for shocks:", list(distance_matrices.keys()))


Saved matrix: country_profile_distance_matrix_shock_2007.csv
Saved matrix: country_profile_similarity_matrix_shock_2007.csv
Saved matrix: country_profile_distance_matrix_shock_2019.csv
Saved matrix: country_profile_similarity_matrix_shock_2019.csv
Matrices created for shocks: ['2007', '2019']


In [7]:

# ------------------------------------------------------
# NEAREST NEIGHBOURS AND EXTREME PAIRS
# ------------------------------------------------------

nearest_rows = []
extreme_rows = []

for shock_id, D in distance_matrices.items():
    for country in D.index:
        temp = (
            D.loc[country]
            .drop(index=country)
            .sort_values(ascending=True)
            .reset_index()
        )
        temp.columns = ["neighbour_country", "ordinal_manhattan_distance_0_1"]
        temp["rank"] = np.arange(1, len(temp) + 1)
        temp["shock_id"] = shock_id
        temp["Country"] = country
        temp["ordinal_similarity_0_1"] = 1 - temp["ordinal_manhattan_distance_0_1"]
        nearest_rows.append(temp[temp["rank"] <= 5])

    pairs = country_profile_distance_long[
        country_profile_distance_long["shock_id"].astype(str) == str(shock_id)
    ].copy()

    most_similar = pairs.sort_values("ordinal_manhattan_distance_0_1", ascending=True).head(10).copy()
    most_similar["pair_type"] = "most_similar"

    most_distant = pairs.sort_values("ordinal_manhattan_distance_0_1", ascending=False).head(10).copy()
    most_distant["pair_type"] = "most_distant"

    extreme_rows.append(pd.concat([most_similar, most_distant], ignore_index=True))

nearest_neighbours_top5 = pd.concat(nearest_rows, ignore_index=True)
country_profile_extreme_pairs = pd.concat(extreme_rows, ignore_index=True)

save_table(
    nearest_neighbours_top5,
    "nearest_neighbours_top5.csv",
    "Nearest neighbours top 5",
    "Closest countries to each country based on final five-variable ordinal profile distance.",
)

save_table(
    country_profile_extreme_pairs,
    "country_profile_extreme_pairs.csv",
    "Most similar and most distant country pairs",
    "Ten most similar and ten most distant country pairs by shock.",
)

display(nearest_neighbours_top5.head(20))
display(country_profile_extreme_pairs.head(20))


Saved table: nearest_neighbours_top5.csv
Saved table: country_profile_extreme_pairs.csv


,neighbour_country,ordinal_manhattan_distance_0_1,rank,shock_id,Country,ordinal_similarity_0_1
0,USA,0.2000,1,2007,AUT,0.8000
1,FRA,0.2000,2,2007,AUT,0.8000
2,SWE,0.2500,3,2007,AUT,0.7500
3,NLD,0.3000,4,2007,AUT,0.7000
4,FIN,0.3000,5,2007,AUT,0.7000
5,FRA,0.2000,1,2007,BEL,0.8000
6,PRT,0.2500,2,2007,BEL,0.7500
7,AUT,0.3000,3,2007,BEL,0.7000
8,DEU,0.3000,4,2007,BEL,0.7000
9,ESP,0.3000,5,2007,BEL,0.7000


,shock_id,country_a,country_b,country_pair,ordinal_manhattan_distance_raw,ordinal_manhattan_distance_0_1,ordinal_similarity_0_1,ordinal_euclidean_distance,dimensions_different_count,dimensions_same_count,abs_diff_Debt capacity,abs_diff_Employment strength,abs_diff_R&D intensity,abs_diff_Tertiary human capital,abs_diff_Energy security,continuous_score_euclidean_distance_0_1,continuous_score_similarity_0_1,pair_type
0,2007,ITA,PRT,ITA-PRT,2.0000,0.1000,0.9000,1.4142,2,3,0.0000,1.0000,0.0000,0.0000,1.0000,0.1758,0.8242,most_similar
1,2007,DEU,FRA,DEU-FRA,2.0000,0.1000,0.9000,1.4142,2,3,0.0000,1.0000,0.0000,1.0000,0.0000,0.0814,0.9186,most_similar
2,2007,SWE,USA,SWE-USA,3.0000,0.1500,0.8500,1.7321,3,2,1.0000,1.0000,0.0000,1.0000,0.0000,0.1912,0.8088,most_similar
3,2007,FIN,SWE,FIN-SWE,3.0000,0.1500,0.8500,1.7321,3,2,0.0000,1.0000,0.0000,1.0000,1.0000,0.0911,0.9089,most_similar
4,2007,HUN,ITA,HUN-ITA,3.0000,0.1500,0.8500,1.7321,3,2,1.0000,1.0000,0.0000,0.0000,1.0000,0.2079,0.7921,most_similar
5,2007,DNK,NLD,DNK-NLD,3.0000,0.1500,0.8500,1.7321,3,2,1.0000,0.0000,1.0000,0.0000,1.0000,0.2544,0.7456,most_similar
6,2007,HUN,PRT,HUN-PRT,3.0000,0.1500,0.8500,2.2361,2,3,1.0000,0.0000,0.0000,0.0000,2.0000,0.1036,0.8964,most_similar
7,2007,NLD,SWE,NLD-SWE,3.0000,0.1500,0.8500,2.2361,2,3,0.0000,2.0000,1.0000,0.0000,0.0000,0.2726,0.7274,most_similar
8,2007,CAN,SWE,CAN-SWE,3.0000,0.1500,0.8500,1.7321,3,2,0.0000,0.0000,1.0000,1.0000,1.0000,0.3959,0.6041,most_similar
9,2007,CAN,NLD,CAN-NLD,4.0000,0.2000,0.8000,2.4495,3,2,0.0000,2.0000,0.0000,1.0000,1.0000,0.3811,0.6189,most_similar


In [8]:

# ------------------------------------------------------
# DISTANCE SUMMARY BY SHOCK
# ------------------------------------------------------

distance_summary_by_shock = (
    country_profile_distance_long
    .groupby("shock_id")
    .agg(
        country_pairs=("country_pair", "count"),
        mean_distance=("ordinal_manhattan_distance_0_1", "mean"),
        median_distance=("ordinal_manhattan_distance_0_1", "median"),
        min_distance=("ordinal_manhattan_distance_0_1", "min"),
        max_distance=("ordinal_manhattan_distance_0_1", "max"),
        mean_similarity=("ordinal_similarity_0_1", "mean"),
        median_similarity=("ordinal_similarity_0_1", "median"),
        mean_dimensions_different=("dimensions_different_count", "mean"),
    )
    .reset_index()
)

save_table(
    distance_summary_by_shock,
    "distance_summary_by_shock.csv",
    "Distance summary by shock",
    "Summary statistics for country-profile distances by shock.",
)

display(distance_summary_by_shock)


Saved table: distance_summary_by_shock.csv


,shock_id,country_pairs,mean_distance,median_distance,min_distance,max_distance,mean_similarity,median_similarity,mean_dimensions_different
0,2007,300,0.4167,0.4000,0.1000,0.8000,0.5833,0.6000,4.1667
1,2019,595,0.4118,0.4000,0.0000,0.9000,0.5882,0.6000,4.1176


In [9]:

# ------------------------------------------------------
# OPTIONAL: DISTANCE BY FRONTIER AND LAYER, IF AVAILABLE
# ------------------------------------------------------

distance_with_positions = country_profile_distance_long.copy()

if "layer" not in country_profiles.columns and "poset_layer" in country_profiles.columns:
    country_profiles["layer"] = country_profiles["poset_layer"]

if "is_pareto_profile" not in country_profiles.columns:
    country_profiles["is_pareto_profile"] = np.nan

position_lookup = country_profiles[
    ["shock_id", "Country"] + [c for c in ["layer", "is_pareto_profile"] if c in country_profiles.columns]
].copy()

position_lookup_a = position_lookup.rename(columns={
    "Country": "country_a",
    "layer": "layer_a",
    "is_pareto_profile": "frontier_a",
})

position_lookup_b = position_lookup.rename(columns={
    "Country": "country_b",
    "layer": "layer_b",
    "is_pareto_profile": "frontier_b",
})

distance_with_positions = distance_with_positions.merge(position_lookup_a, on=["shock_id", "country_a"], how="left")
distance_with_positions = distance_with_positions.merge(position_lookup_b, on=["shock_id", "country_b"], how="left")

if "layer_a" in distance_with_positions.columns:
    distance_with_positions["same_layer"] = (
        distance_with_positions["layer_a"].astype(str)
        == distance_with_positions["layer_b"].astype(str)
    )

if "frontier_a" in distance_with_positions.columns:
    distance_with_positions["frontier_pair_type"] = np.select(
        [
            (distance_with_positions["frontier_a"] == True) & (distance_with_positions["frontier_b"] == True),
            (distance_with_positions["frontier_a"] == False) & (distance_with_positions["frontier_b"] == False),
            distance_with_positions["frontier_a"].notna() & distance_with_positions["frontier_b"].notna(),
        ],
        [
            "frontier_frontier",
            "nonfrontier_nonfrontier",
            "mixed_frontier_nonfrontier",
        ],
        default="unknown",
    )

save_table(
    distance_with_positions,
    "country_profile_distance_with_poset_positions.csv",
    "Country profile distance with POSet positions",
    "Pairwise distances enriched with POSet layer and frontier information where available.",
)

if "same_layer" in distance_with_positions.columns:
    distance_by_same_layer = (
        distance_with_positions
        .groupby(["shock_id", "same_layer"])
        .agg(
            pairs=("country_pair", "count"),
            mean_distance=("ordinal_manhattan_distance_0_1", "mean"),
            median_distance=("ordinal_manhattan_distance_0_1", "median"),
            mean_similarity=("ordinal_similarity_0_1", "mean"),
        )
        .reset_index()
    )

    save_table(
        distance_by_same_layer,
        "distance_by_same_layer_summary.csv",
        "Distance by same-layer status",
        "Compares profile distance for same-layer versus different-layer country pairs.",
    )

if "frontier_pair_type" in distance_with_positions.columns:
    distance_by_frontier_type = (
        distance_with_positions
        .groupby(["shock_id", "frontier_pair_type"])
        .agg(
            pairs=("country_pair", "count"),
            mean_distance=("ordinal_manhattan_distance_0_1", "mean"),
            median_distance=("ordinal_manhattan_distance_0_1", "median"),
            mean_similarity=("ordinal_similarity_0_1", "mean"),
        )
        .reset_index()
    )

    save_table(
        distance_by_frontier_type,
        "distance_by_frontier_pair_type_summary.csv",
        "Distance by frontier pair type",
        "Compares profile distance among frontier-frontier, non-frontier, and mixed pairs.",
    )

display(distance_with_positions.head())


Saved table: country_profile_distance_with_poset_positions.csv
Saved table: distance_by_same_layer_summary.csv
Saved table: distance_by_frontier_pair_type_summary.csv


,shock_id,country_a,country_b,country_pair,ordinal_manhattan_distance_raw,ordinal_manhattan_distance_0_1,ordinal_similarity_0_1,ordinal_euclidean_distance,dimensions_different_count,dimensions_same_count,abs_diff_Debt capacity,abs_diff_Employment strength,abs_diff_R&D intensity,abs_diff_Tertiary human capital,abs_diff_Energy security,continuous_score_euclidean_distance_0_1,continuous_score_similarity_0_1,layer_a,frontier_a,layer_b,frontier_b,same_layer,frontier_pair_type
0,2007,AUT,BEL,AUT-BEL,6.0000,0.3000,0.7000,2.8284,5,0,1.0000,2.0000,1.0000,1.0000,1.0000,0.2284,0.7716,2,NaN,3,NaN,False,unknown
1,2007,AUT,CAN,AUT-CAN,8.0000,0.4000,0.6000,4.0000,5,0,1.0000,1.0000,1.0000,2.0000,3.0000,0.4850,0.5150,2,NaN,1,NaN,False,unknown
2,2007,AUT,CZE,AUT-CZE,10.0000,0.5000,0.5000,4.6904,5,0,2.0000,1.0000,2.0000,2.0000,3.0000,0.3132,0.6868,2,NaN,2,NaN,True,unknown
3,2007,AUT,DEU,AUT-DEU,6.0000,0.3000,0.7000,3.4641,4,1,0.0000,3.0000,1.0000,1.0000,1.0000,0.2232,0.7768,2,NaN,4,NaN,False,unknown
4,2007,AUT,DNK,AUT-DNK,7.0000,0.3500,0.6500,3.8730,4,1,2.0000,1.0000,0.0000,1.0000,3.0000,0.3340,0.6660,2,NaN,1,NaN,False,unknown


In [10]:

# ------------------------------------------------------
# FIGURE: ORDINAL DISTANCE HEATMAPS
# ------------------------------------------------------

def plot_matrix_heatmap(matrix, title, file_name, cmap="viridis_r", vmin=0, vmax=1):
    fig, ax = plt.subplots(figsize=(11, 9))

    im = ax.imshow(matrix.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_xticklabels(matrix.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(matrix.index, fontsize=7)

    ax.set_title(title, fontsize=14, pad=14)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Normalized ordinal distance", fontsize=10)

    ax.set_xlabel("Country")
    ax.set_ylabel("Country")

    plt.tight_layout()

    save_figure(
        fig,
        file_name,
        title,
        "Heatmap of normalized ordinal Manhattan distances across final five-variable profiles.",
    )

for shock_id, D in distance_matrices.items():
    plot_matrix_heatmap(
        D,
        f"Country profile distance heatmap — {shock_id}",
        f"19_country_profile_distance_heatmap_shock_{shock_id}.png",
        cmap="viridis_r",
        vmin=0,
        vmax=1,
    )


Saved figure: 19_country_profile_distance_heatmap_shock_2007.png
Saved figure: 19_country_profile_distance_heatmap_shock_2019.png


In [11]:

# ------------------------------------------------------
# FIGURE: SIMILARITY HEATMAPS
# ------------------------------------------------------

def plot_similarity_heatmap(matrix, title, file_name, cmap="viridis", vmin=0, vmax=1):
    fig, ax = plt.subplots(figsize=(11, 9))

    im = ax.imshow(matrix.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_xticklabels(matrix.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(matrix.index, fontsize=7)

    ax.set_title(title, fontsize=14, pad=14)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Ordinal similarity = 1 - distance", fontsize=10)

    ax.set_xlabel("Country")
    ax.set_ylabel("Country")

    plt.tight_layout()

    save_figure(
        fig,
        file_name,
        title,
        "Heatmap of ordinal similarities across final five-variable profiles.",
    )

for shock_id, S in similarity_matrices.items():
    plot_similarity_heatmap(
        S,
        f"Country profile similarity heatmap — {shock_id}",
        f"19_country_profile_similarity_heatmap_shock_{shock_id}.png",
        cmap="viridis",
        vmin=0,
        vmax=1,
    )


Saved figure: 19_country_profile_similarity_heatmap_shock_2007.png
Saved figure: 19_country_profile_similarity_heatmap_shock_2019.png


In [12]:

# ------------------------------------------------------
# CLASSICAL MDS FROM DISTANCE MATRIX
# ------------------------------------------------------

def classical_mds(distance_matrix, n_components=2):
    # Classical metric multidimensional scaling using eigen-decomposition.
    D = distance_matrix.values.astype(float)
    n = D.shape[0]

    D2 = D ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J

    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]

    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    positive = eigvals > 1e-12
    eigvals_pos = eigvals[positive][:n_components]
    eigvecs_pos = eigvecs[:, positive][:, :n_components]

    coords = eigvecs_pos * np.sqrt(eigvals_pos)

    if coords.shape[1] < n_components:
        coords = np.column_stack([coords, np.zeros((n, n_components - coords.shape[1]))])

    return pd.DataFrame(
        coords,
        index=distance_matrix.index,
        columns=[f"mds_dim_{i+1}" for i in range(n_components)],
    ), eigvals

mds_rows = []

for shock_id, D in distance_matrices.items():
    coords, eigvals = classical_mds(D, n_components=2)
    coords = coords.reset_index().rename(columns={"index": "Country"})
    coords["shock_id"] = shock_id

    meta_cols = ["shock_id", "Country", "country_name"]
    for c in ["layer", "poset_layer", "is_pareto_profile", "profile_code"]:
        if c in country_profiles.columns:
            meta_cols.append(c)

    meta = country_profiles[meta_cols].drop_duplicates(["shock_id", "Country"]).copy()
    coords = coords.merge(meta, on=["shock_id", "Country"], how="left")
    mds_rows.append(coords)

profile_distance_mds_coordinates = pd.concat(mds_rows, ignore_index=True)

save_table(
    profile_distance_mds_coordinates,
    "profile_distance_mds_coordinates.csv",
    "Profile distance MDS coordinates",
    "Two-dimensional classical MDS coordinates from ordinal profile distances.",
)

display(profile_distance_mds_coordinates.head())


Saved table: profile_distance_mds_coordinates.csv


,Country,mds_dim_1,mds_dim_2,shock_id,country_name,layer,is_pareto_profile,profile_code
0,AUT,-0.0515,0.1344,2007,AUT,2,NaN,2-4-5-3-2
1,BEL,0.0682,0.2839,2007,BEL,3,NaN,1-2-4-4-1
2,CAN,-0.2529,0.1082,2007,CAN,1,NaN,3-3-4-5-5
3,CZE,-0.0140,-0.1196,2007,CZE,2,NaN,4-3-3-1-5
4,DEU,0.1360,0.1948,2007,DEU,4,NaN,2-1-4-2-3


In [13]:

# ------------------------------------------------------
# FIGURE: MDS SIMILARITY MAPS
# ------------------------------------------------------

for shock_id, coords in profile_distance_mds_coordinates.groupby("shock_id"):
    fig, ax = plt.subplots(figsize=(10, 7))

    layer_col = None
    if "layer" in coords.columns and coords["layer"].notna().any():
        layer_col = "layer"
    elif "poset_layer" in coords.columns and coords["poset_layer"].notna().any():
        layer_col = "poset_layer"

    if layer_col:
        color_values = pd.to_numeric(coords[layer_col], errors="coerce")
        scatter = ax.scatter(
            coords["mds_dim_1"],
            coords["mds_dim_2"],
            c=color_values,
            s=120,
            edgecolor="black",
            linewidth=0.8,
            cmap="viridis_r",
        )
        cbar = fig.colorbar(scatter, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("POSet layer", fontsize=10)
    else:
        ax.scatter(
            coords["mds_dim_1"],
            coords["mds_dim_2"],
            s=120,
            edgecolor="black",
            linewidth=0.8,
        )

    for _, row in coords.iterrows():
        ax.text(
            row["mds_dim_1"],
            row["mds_dim_2"],
            row["Country"],
            fontsize=8,
            ha="center",
            va="center",
        )

    ax.axhline(0, linewidth=0.8, alpha=0.3)
    ax.axvline(0, linewidth=0.8, alpha=0.3)
    ax.set_title(f"Country similarity map from profile distances — {shock_id}", fontsize=14, pad=14)
    ax.set_xlabel("MDS dimension 1")
    ax.set_ylabel("MDS dimension 2")
    ax.grid(True, alpha=0.25)

    ax.text(
        0.01,
        -0.12,
        "Countries closer together have more similar five-variable ordinal structural profiles.",
        transform=ax.transAxes,
        fontsize=9,
        ha="left",
    )

    plt.tight_layout()

    save_figure(
        fig,
        f"19_country_profile_similarity_mds_map_shock_{shock_id}.png",
        f"Country profile similarity MDS map shock {shock_id}",
        "Two-dimensional map of country similarities from final five-variable ordinal profile distances.",
    )


Saved figure: 19_country_profile_similarity_mds_map_shock_2007.png
Saved figure: 19_country_profile_similarity_mds_map_shock_2019.png


In [14]:

# ------------------------------------------------------
# REPORT-READY NOTES
# ------------------------------------------------------

report_ready_similarity_distance_notes = pd.DataFrame([
    {
        "section": "Methodology",
        "note_id": "profile_distance_definition",
        "report_text": (
            "To complement the dominance-based POSet, pairwise similarity distances were computed across the final "
            "five-variable ordinal country profiles. For each pair of countries, the distance is the normalized sum "
            "of absolute differences across debt capacity, employment strength, R&D intensity, tertiary human capital "
            "and energy security. A distance of 0 indicates identical ordinal profiles, while a distance of 1 indicates "
            "maximum difference across the five dimensions."
        ),
    },
    {
        "section": "Results",
        "note_id": "profile_distance_interpretation",
        "report_text": (
            "The similarity-distance analysis should not be interpreted as a replacement for the POSet. The POSet "
            "identifies dominance, frontier positions and incomparability, while the distance matrix shows how close "
            "countries are in structural-profile space. This is especially useful for comparing countries that are "
            "incomparable under Pareto dominance but still structurally similar."
        ),
    },
    {
        "section": "Appendix",
        "note_id": "nearest_neighbour_use",
        "report_text": (
            "The nearest-neighbour table identifies, for each country, the countries with the most similar final "
            "five-variable structural profile. This can support interpretation of comparable reform groups without "
            "turning the POSet into a scalar resilience ranking."
        ),
    },
])

methodological_decision_similarity_distance = pd.DataFrame([
    {
        "decision": "Add pairwise similarity-distance matrix",
        "final_choice": "Normalized ordinal Manhattan distance across final five-variable profiles",
        "reason": (
            "This measures structural closeness without replacing the POSet dominance relation or forcing a scalar resilience index."
        ),
        "alternatives_rejected": (
            "Using only raw continuous variables; adding WGI/governance as a main distance dimension; using subjective weights."
        ),
    },
    {
        "decision": "Use five final ordering dimensions only",
        "final_choice": "Debt capacity, employment strength, R&D intensity, tertiary human capital, energy security",
        "reason": "Keeps distance analysis coherent with the final baseline POSet.",
        "alternatives_rejected": "Adding WGI/governance or volatility to the main similarity distance.",
    },
])

save_table(
    report_ready_similarity_distance_notes,
    "report_ready_similarity_distance_notes.csv",
    "Report-ready similarity distance notes",
    "Text snippets explaining the similarity-distance method and interpretation.",
)

save_table(
    methodological_decision_similarity_distance,
    "methodological_decision_similarity_distance.csv",
    "Methodological decision similarity distance",
    "Decision log entries for the profile similarity-distance extension.",
)

display(report_ready_similarity_distance_notes)
display(methodological_decision_similarity_distance)


Saved table: report_ready_similarity_distance_notes.csv
Saved table: methodological_decision_similarity_distance.csv


,section,note_id,report_text
0,Methodology,profile_distance_definition,"To complement the dominance-based POSet, pairw..."
1,Results,profile_distance_interpretation,The similarity-distance analysis should not be...
2,Appendix,nearest_neighbour_use,"The nearest-neighbour table identifies, for ea..."


,decision,final_choice,reason,alternatives_rejected
0,Add pairwise similarity-distance matrix,Normalized ordinal Manhattan distance across f...,This measures structural closeness without rep...,Using only raw continuous variables; adding WG...
1,Use five final ordering dimensions only,"Debt capacity, employment strength, R&D intens...",Keeps distance analysis coherent with the fina...,Adding WGI/governance or volatility to the mai...


In [15]:

# ------------------------------------------------------
# AUDIT WORKBOOK AND FINAL INVENTORIES
# ------------------------------------------------------

profile_similarity_distance_run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "created_at": RUN_TIMESTAMP,
    "main_variable_set": MAIN_VARIABLE_SET_NAME,
    "main_levels": MAIN_LEVELS,
    "distance_method": "normalized ordinal Manhattan distance",
    "similarity_method": "1 - normalized ordinal Manhattan distance",
    "ordering_variables_count": len(level_columns),
    "shocks": ", ".join(sorted(country_profiles["shock_id"].astype(str).unique())),
    "country_rows": len(country_profiles),
    "pairwise_distance_rows": len(country_profile_distance_long),
    "figures_created": len(figure_inventory_rows),
    "tables_created": len(table_inventory_rows),
    "note": "Similarity-distance analysis complements POSet dominance; it does not create a scalar resilience index.",
}])

save_table(
    profile_similarity_distance_run_summary,
    "profile_similarity_distance_run_summary.csv",
    "Profile similarity distance run summary",
    "Run summary for pairwise country/profile similarity-distance analysis.",
)

figure_inventory = pd.DataFrame(figure_inventory_rows)
table_inventory = pd.DataFrame(table_inventory_rows)

figure_inventory.to_csv(PROCESSED_DIR / "profile_similarity_distance_figure_inventory.csv", index=False)
figure_inventory.to_csv(DIAGNOSTICS_DIR / "profile_similarity_distance_figure_inventory.csv", index=False)
figure_inventory.to_csv(TABLES_DIR / "profile_similarity_distance_figure_inventory.csv", index=False)

table_inventory.to_csv(PROCESSED_DIR / "profile_similarity_distance_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "profile_similarity_distance_table_inventory.csv", index=False)
table_inventory.to_csv(TABLES_DIR / "profile_similarity_distance_table_inventory.csv", index=False)

audit_xlsx = AUDIT_DIR / "19_Profile_Similarity_Distance_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    profile_similarity_distance_run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    distance_summary_by_shock.to_excel(writer, sheet_name="distance_summary", index=False)
    nearest_neighbours_top5.to_excel(writer, sheet_name="nearest_neighbours", index=False)
    country_profile_extreme_pairs.to_excel(writer, sheet_name="extreme_pairs", index=False)
    report_ready_similarity_distance_notes.to_excel(writer, sheet_name="report_notes", index=False)
    methodological_decision_similarity_distance.to_excel(writer, sheet_name="decision_log", index=False)
    figure_inventory.to_excel(writer, sheet_name="figure_inventory", index=False)
    table_inventory.to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook:", audit_xlsx.resolve())
display(profile_similarity_distance_run_summary)
display(figure_inventory)
display(table_inventory)


Saved table: profile_similarity_distance_run_summary.csv
Audit workbook: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\19_Profile_Similarity_Distance_Audit.xlsx


,run_id,created_at,main_variable_set,main_levels,distance_method,similarity_method,ordering_variables_count,shocks,country_rows,pairwise_distance_rows,figures_created,tables_created,note
0,20260625_094100,2026-06-25 09:41:00,baseline_5_variables,5,normalized ordinal Manhattan distance,1 - normalized ordinal Manhattan distance,5,"2007, 2019",60,895,6,14,Similarity-distance analysis complements POSet...


,file_name,title,description,path,created_at
0,19_country_profile_distance_heatmap_shock_2007...,Country profile distance heatmap — 2007,Heatmap of normalized ordinal Manhattan distan...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
1,19_country_profile_distance_heatmap_shock_2019...,Country profile distance heatmap — 2019,Heatmap of normalized ordinal Manhattan distan...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
2,19_country_profile_similarity_heatmap_shock_20...,Country profile similarity heatmap — 2007,Heatmap of ordinal similarities across final f...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
3,19_country_profile_similarity_heatmap_shock_20...,Country profile similarity heatmap — 2019,Heatmap of ordinal similarities across final f...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
4,19_country_profile_similarity_mds_map_shock_20...,Country profile similarity MDS map shock 2007,Two-dimensional map of country similarities fr...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
5,19_country_profile_similarity_mds_map_shock_20...,Country profile similarity MDS map shock 2019,Two-dimensional map of country similarities fr...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00


,file_name,title,description,rows,columns,processed_path,diagnostics_path,table_path,created_at
0,country_profile_distance_long.csv,Country profile pairwise distance long table,Pairwise ordinal profile distances and similar...,895,17,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
1,country_profile_distance_matrix_shock_2007.csv,Country profile distance matrix shock 2007,Normalized ordinal Manhattan distance matrix.,25,25,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
2,country_profile_similarity_matrix_shock_2007.csv,Country profile similarity matrix shock 2007,Ordinal profile similarity matrix where simila...,25,25,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
3,country_profile_distance_matrix_shock_2019.csv,Country profile distance matrix shock 2019,Normalized ordinal Manhattan distance matrix.,35,35,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
4,country_profile_similarity_matrix_shock_2019.csv,Country profile similarity matrix shock 2019,Ordinal profile similarity matrix where simila...,35,35,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
5,nearest_neighbours_top5.csv,Nearest neighbours top 5,Closest countries to each country based on fin...,300,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
6,country_profile_extreme_pairs.csv,Most similar and most distant country pairs,Ten most similar and ten most distant country ...,40,18,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
7,distance_summary_by_shock.csv,Distance summary by shock,Summary statistics for country-profile distanc...,2,9,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
8,country_profile_distance_with_poset_positions.csv,Country profile distance with POSet positions,Pairwise distances enriched with POSet layer a...,895,23,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00
9,distance_by_same_layer_summary.csv,Distance by same-layer status,Compares profile distance for same-layer versu...,4,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-25 09:41:00


In [16]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

expected_core_outputs = [
    "country_profile_distance_long.csv",
    "nearest_neighbours_top5.csv",
    "country_profile_extreme_pairs.csv",
    "distance_summary_by_shock.csv",
    "profile_distance_mds_coordinates.csv",
    "report_ready_similarity_distance_notes.csv",
    "methodological_decision_similarity_distance.csv",
    "profile_similarity_distance_run_summary.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_core_outputs
])

figure_check = pd.DataFrame([
    {
        "file_name": row["file_name"],
        "exists": Path(row["path"]).exists(),
        "path": row["path"],
    }
    for row in figure_inventory_rows
])

output_check.to_csv(DIAGNOSTICS_DIR / "similarity_distance_output_check.csv", index=False)
figure_check.to_csv(DIAGNOSTICS_DIR / "similarity_distance_figure_check.csv", index=False)

print("19 PROFILE SIMILARITY DISTANCE COMPLETE")
print("=" * 90)

display(output_check)
display(figure_check)

print("\nWhat to tell Francesco:")
print("Yes — we added a similarity-distance module.")
print("It computes pairwise distances across the final five-variable ordinal country profiles.")
print("The main distance is normalized Manhattan distance; similarity is 1 - distance.")
print("This complements POSet dominance and does not create a scalar Economic Resilience Index.")

print("\nNext outputs to send/check:")
print("- 19_Profile_Similarity_Distance_Audit.xlsx")
print("- country_profile_distance_long.csv")
print("- nearest_neighbours_top5.csv")
print("- country_profile_extreme_pairs.csv")
print("- 19_country_profile_distance_heatmap_shock_2007.png")
print("- 19_country_profile_distance_heatmap_shock_2019.png")
print("- 19_country_profile_similarity_mds_map_shock_2007.png")
print("- 19_country_profile_similarity_mds_map_shock_2019.png")


19 PROFILE SIMILARITY DISTANCE COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,country_profile_distance_long.csv,True,True,True
1,nearest_neighbours_top5.csv,True,True,True
2,country_profile_extreme_pairs.csv,True,True,True
3,distance_summary_by_shock.csv,True,True,True
4,profile_distance_mds_coordinates.csv,True,True,True
5,report_ready_similarity_distance_notes.csv,True,True,True
6,methodological_decision_similarity_distance.csv,True,True,True
7,profile_similarity_distance_run_summary.csv,True,True,True


,file_name,exists,path
0,19_country_profile_distance_heatmap_shock_2007...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
1,19_country_profile_distance_heatmap_shock_2019...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
2,19_country_profile_similarity_heatmap_shock_20...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
3,19_country_profile_similarity_heatmap_shock_20...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
4,19_country_profile_similarity_mds_map_shock_20...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
5,19_country_profile_similarity_mds_map_shock_20...,True,D:\Milano Bicocca\Course Materials\1st Year\02...



What to tell Francesco:
Yes — we added a similarity-distance module.
It computes pairwise distances across the final five-variable ordinal country profiles.
The main distance is normalized Manhattan distance; similarity is 1 - distance.
This complements POSet dominance and does not create a scalar Economic Resilience Index.

Next outputs to send/check:
- 19_Profile_Similarity_Distance_Audit.xlsx
- country_profile_distance_long.csv
- nearest_neighbours_top5.csv
- country_profile_extreme_pairs.csv
- 19_country_profile_distance_heatmap_shock_2007.png
- 19_country_profile_distance_heatmap_shock_2019.png
- 19_country_profile_similarity_mds_map_shock_2007.png
- 19_country_profile_similarity_mds_map_shock_2019.png
